# Bank Customer Churn — Notebook Final

Solución completa para la competencia de Kaggle de **clasificación de churn bancario**.

**Métrica de evaluación:** ROC-AUC

## Índice
1. Carga de datos y exploración inicial (EDA)
2. Limpieza: nulos, duplicados, outliers
3. Preprocesamiento y **feature engineering**
4. Modelos base (Logistic Regression, Random Forest, XGBoost, LightGBM)
5. Optimización con **RandomizedSearchCV**
6. Solución final: **ensemble multi-seed** (XGBoost + LightGBM + CatBoost + HistGB)
7. Generación del `submission.csv` y conclusiones

La estrategia ganadora combina **feature engineering**, **K-Fold con test averaging**, **multi-seed averaging** y un **blend ponderado** de cuatro modelos heterogéneos.

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, classification_report, roc_curve

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, early_stopping as lgb_early_stop
from catboost import CatBoostClassifier
from scipy.stats import randint, uniform

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
pd.set_option('display.max_columns', 60)
SEED = 42
np.random.seed(SEED)

## 1. Carga y exploración inicial (EDA)

El dataset describe clientes de un banco con variables demográficas (`Age`, `Gender`, `Geography`) y financieras (`CreditScore`, `Balance`, `EstimatedSalary`, `NumOfProducts`, `HasCrCard`, `IsActiveMember`, `Tenure`). La variable objetivo `Exited` indica si el cliente abandonó la entidad.

In [ ]:
DATA_DIR = '.'  # <-- ajusta si tus CSV están en otra carpeta
train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
sample = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))

print('Train:', train.shape, ' Test:', test.shape, ' Sample sub:', sample.shape)
train.head()

In [ ]:
train.info()
print('\nEstadísticas descriptivas:')
train.describe(include='all').T

In [ ]:
# Distribución de la clase objetivo
print('Distribución de Exited:')
print(train['Exited'].value_counts(normalize=True))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x='Exited', data=train, ax=axes[0])
axes[0].set_title('Conteo de churn (Exited)')
sns.histplot(data=train, x='Age', hue='Exited', bins=30, kde=True, ax=axes[1])
axes[1].set_title('Edad por churn')
plt.tight_layout(); plt.show()

In [ ]:
# Análisis por variables categóricas
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(x='Geography', y='Exited', data=train, ax=axes[0])
axes[0].set_title('Tasa de churn por Geography')
sns.barplot(x='Gender', y='Exited', data=train, ax=axes[1])
axes[1].set_title('Tasa de churn por Gender')
plt.tight_layout(); plt.show()

num_cols_corr = ['CreditScore','Age','Tenure','Balance','NumOfProducts','HasCrCard','IsActiveMember','EstimatedSalary','Exited']
plt.figure(figsize=(9,6))
sns.heatmap(train[num_cols_corr].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Matriz de correlaciones'); plt.show()

**Hallazgos del EDA**

- La clase está **desbalanceada**: ~20% de churn.
- **Age** es la variable más correlacionada con el target: clientes mayores abandonan más.
- **Geography**: Alemania presenta una tasa de churn notablemente mayor que Francia o España.
- **Gender**: las mujeres muestran ligeramente más churn que los hombres.
- `IsActiveMember`, `Balance` y `NumOfProducts` también aportan señal.
- `id`, `CustomerId` y `Surname` son identificadores; los descartamos del modelo (aunque la frecuencia del apellido sí la usamos como feature).

## 2. Limpieza: nulos, duplicados y outliers

In [ ]:
print('Nulos en train:'); print(train.isna().sum())
print('\nNulos en test :'); print(test.isna().sum())
print('\nDuplicados train:', train.duplicated().sum())
print('Duplicados test :', test.duplicated().sum())

In [ ]:
# Imputación: numéricas -> mediana, categóricas -> moda
num_cols = ['CreditScore','Age','Tenure','Balance','NumOfProducts','HasCrCard','IsActiveMember','EstimatedSalary']
cat_cols = ['Geography','Gender','Surname']

for df in (train, test):
    for c in num_cols:
        if df[c].isna().any(): df[c] = df[c].fillna(df[c].median())
    for c in cat_cols:
        if df[c].isna().any(): df[c] = df[c].fillna(df[c].mode().iloc[0])

train = train.dropna(subset=['Exited']).reset_index(drop=True)
train['Exited'] = train['Exited'].astype(int)
train = train.drop_duplicates().reset_index(drop=True)

# Outliers - boxplots
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for ax, c in zip(axes.ravel(), num_cols):
    sns.boxplot(x=train[c], ax=ax); ax.set_title(c)
plt.tight_layout(); plt.show()

# Winsorización suave (clip a percentiles extremos)
def clip_outliers(df, col, lo=0.001, hi=0.999):
    a, b = df[col].quantile([lo, hi])
    df[col] = df[col].clip(a, b)
    return df
for c in ['Age','CreditScore']:
    train = clip_outliers(train, c)
    test  = clip_outliers(test, c)

print('Train shape tras limpieza:', train.shape)

## 3. Preprocesamiento y Feature Engineering

Aquí está la mayor palanca para subir el AUC. Construimos:

- **Ratios**: `BalanceSalaryRatio`, `CreditScoreAge`, `TenureByAge`, `BalancePerProduct`, etc.
- **Interacciones**: `AgeBalance`, `BalanceActive`, `Geo_Gender`, `Geo_Age`...
- **Bins**: `AgeGroup`, `CreditGroup`, `BalanceGroup`.
- **Indicadores binarios**: `IsZeroBalance`, `IsSenior`, `Inactive_NoCard`, `ManyProducts`...
- **Frequency encoding** del apellido (sobre train+test, sin target).
- **Target encoding K-Fold** de las variables categóricas (sin leakage).

In [ ]:
def engineer(df):
    df = df.copy()
    df['IsZeroBalance']      = (df['Balance'] == 0).astype(int)
    df['BalanceSalaryRatio'] = df['Balance'] / (df['EstimatedSalary'] + 1)
    df['CreditScoreAge']     = df['CreditScore'] / (df['Age'] + 1)
    df['TenureByAge']        = df['Tenure'] / (df['Age'] + 1)
    df['ProductsPerTenure']  = df['NumOfProducts'] / (df['Tenure'] + 1)
    df['BalancePerProduct']  = df['Balance'] / (df['NumOfProducts'] + 1)
    df['SalaryPerProduct']   = df['EstimatedSalary'] / (df['NumOfProducts'] + 1)
    df['AgeBalance']         = df['Age'] * df['Balance']
    df['AgeCreditScore']     = df['Age'] * df['CreditScore']
    df['AgeProducts']        = df['Age'] * df['NumOfProducts']
    df['BalanceActive']      = df['Balance'] * df['IsActiveMember']
    df['CreditActive']       = df['CreditScore'] * df['IsActiveMember']
    df['ProductsActive']     = df['NumOfProducts'] * df['IsActiveMember']
    df['AgeGroup']           = pd.cut(df['Age'], bins=[0,30,40,50,60,100], labels=[0,1,2,3,4]).astype(int)
    df['CreditGroup']        = pd.cut(df['CreditScore'], bins=[0,580,670,740,800,1000], labels=[0,1,2,3,4]).astype(int)
    df['BalanceGroup']       = pd.cut(df['Balance'], bins=[-1,1,50000,100000,150000,1e9], labels=[0,1,2,3,4]).astype(int)
    df['HighBalance']        = (df['Balance'] > 100000).astype(int)
    df['IsSenior']           = (df['Age'] >= 60).astype(int)
    df['IsYoung']            = (df['Age'] <= 30).astype(int)
    df['Inactive_NoCard']    = ((df['IsActiveMember']==0) & (df['HasCrCard']==0)).astype(int)
    df['Active_HasCard']     = ((df['IsActiveMember']==1) & (df['HasCrCard']==1)).astype(int)
    df['Geo_Gender']         = df['Geography'].astype(str) + '_' + df['Gender'].astype(str)
    df['Geo_Age']            = df['Geography'].astype(str) + '_' + df['AgeGroup'].astype(str)
    df['LowTenure_HighBalance'] = ((df['Tenure']<=2) & (df['Balance']>100000)).astype(int)
    df['ManyProducts']       = (df['NumOfProducts'] >= 3).astype(int)
    df['SingleProduct']      = (df['NumOfProducts'] == 1).astype(int)
    df['SurnameLen']         = df['Surname'].astype(str).str.len()
    return df

train_fe = engineer(train)
test_fe  = engineer(test)

# Frequency encoding del apellido (sin target)
all_surnames = pd.concat([train_fe['Surname'], test_fe['Surname']])
freq_map = all_surnames.value_counts(normalize=True).to_dict()
train_fe['SurnameFreq'] = train_fe['Surname'].map(freq_map).fillna(0)
test_fe['SurnameFreq']  = test_fe['Surname'].map(freq_map).fillna(0)

drop_cols = ['id','CustomerId','Surname']
y        = train_fe['Exited'].values
test_ids = test_fe['id'].values
X      = train_fe.drop(columns=drop_cols + ['Exited'])
X_test = test_fe.drop(columns=drop_cols)

# Target encoding K-fold (sin leakage)
def kfold_te(train_col, test_col, y, n_splits=5, smooth=20):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    enc_tr = np.zeros(len(train_col)); gm = y.mean()
    for tr, va in skf.split(train_col, y):
        s = pd.DataFrame({'c': train_col.iloc[tr].values, 'y': y[tr]}).groupby('c')['y'].agg(['mean','count'])
        s['enc'] = (s['mean']*s['count'] + gm*smooth) / (s['count'] + smooth)
        m = s['enc'].to_dict()
        enc_tr[va] = pd.Series(train_col.iloc[va].values).map(m).fillna(gm).values
    s = pd.DataFrame({'c': train_col.values, 'y': y}).groupby('c')['y'].agg(['mean','count'])
    s['enc'] = (s['mean']*s['count'] + gm*smooth) / (s['count'] + smooth)
    m = s['enc'].to_dict()
    enc_te = pd.Series(test_col.values).map(m).fillna(gm).values
    return enc_tr, enc_te

for c in ['Surname','Geography','Gender','Geo_Gender','Geo_Age']:
    tr_te, te_te = kfold_te(train_fe[c], test_fe[c], y)
    X[f'{c}_te']      = tr_te
    X_test[f'{c}_te'] = te_te

# One-hot encoding del resto de categóricas
X      = pd.get_dummies(X,      columns=['Geography','Gender','Geo_Gender','Geo_Age'], drop_first=False)
X_test = pd.get_dummies(X_test, columns=['Geography','Gender','Geo_Gender','Geo_Age'], drop_first=False)
X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)

print(f'Features finales: {X.shape[1]}')
X.head()

In [ ]:
# Escalado (necesario para Logistic Regression; inocuo para los árboles)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
X_train_sc, X_val_sc, _, _ = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=SEED
)
print('Train:', X_train.shape, ' Val:', X_val.shape)

## 4. Modelos base — comparación con 5-Fold CV

Comparamos cuatro familias de modelos para entender cuál merece la pena tunear y ensamblar:
Logistic Regression (lineal de referencia), Random Forest, XGBoost y LightGBM.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

models_base = {
    'LogisticRegression': (LogisticRegression(max_iter=1000, random_state=SEED), True),
    'RandomForest'      : (RandomForestClassifier(n_estimators=150, random_state=SEED, n_jobs=-1), False),
    'XGBoost'           : (XGBClassifier(n_estimators=250, learning_rate=0.07, max_depth=5,
                                          eval_metric='auc', random_state=SEED, n_jobs=-1), False),
    'LightGBM'          : (LGBMClassifier(n_estimators=300, learning_rate=0.07, num_leaves=31,
                                           random_state=SEED, n_jobs=-1, verbose=-1), False),
}

cv_results = {}
for name, (model, needs_scaling) in models_base.items():
    X_cv = X_scaled if needs_scaling else X.values
    scores = cross_val_score(model, X_cv, y, cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:20s}  ROC-AUC = {scores.mean():.5f}  +/- {scores.std():.5f}')

results_df = pd.DataFrame(cv_results)
plt.figure(figsize=(8,4))
sns.boxplot(data=results_df)
plt.title('ROC-AUC por modelo (5-fold CV)'); plt.ylabel('ROC-AUC'); plt.show()
best_base = results_df.mean().idxmax()
print(f'\nMejor modelo base: {best_base}')

**Conclusión parcial:** los modelos de *gradient boosting* (XGBoost y LightGBM) baten claramente a Logistic Regression y a Random Forest. Vamos a tunear el mejor con `RandomizedSearchCV`.

## 5. Optimización con RandomizedSearchCV

Definimos un grid para el mejor modelo base y dejamos que `RandomizedSearchCV` busque hiperparámetros optimizando `roc_auc` con 3-fold CV.

In [ ]:
param_dist_xgb = {
    'n_estimators'    : randint(300, 1200),
    'learning_rate'   : uniform(0.01, 0.15),
    'max_depth'       : randint(3, 10),
    'min_child_weight': randint(1, 10),
    'subsample'       : uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'gamma'           : uniform(0, 0.5),
    'reg_alpha'       : uniform(0, 1),
    'reg_lambda'      : uniform(0, 1),
}

search = RandomizedSearchCV(
    XGBClassifier(eval_metric='auc', random_state=SEED, n_jobs=-1, tree_method='hist'),
    param_distributions=param_dist_xgb,
    n_iter=15, scoring='roc_auc',
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED),
    n_jobs=-1, random_state=SEED, verbose=1
)
search.fit(X.values, y)
print(f'Mejor ROC-AUC CV: {search.best_score_:.5f}')
print('Mejores hiperparámetros:'); print(search.best_params_)

In [ ]:
# Evaluación del XGBoost tuneado en validación
best_xgb = search.best_estimator_
best_xgb.fit(X_train.values, y_train)
val_proba = best_xgb.predict_proba(X_val.values)[:,1]
val_auc = roc_auc_score(y_val, val_proba)
print(f'ROC-AUC validación XGBoost tuneado: {val_auc:.5f}\n')
print(classification_report(y_val, (val_proba>=0.5).astype(int)))

fpr, tpr, _ = roc_curve(y_val, val_proba)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f'AUC = {val_auc:.4f}')
plt.plot([0,1],[0,1],'--', color='gray')
plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('Curva ROC - XGBoost tuneado')
plt.legend(); plt.show()

## 6. Solución final: Ensemble multi-seed (XGB + LGB + CatBoost + HistGB)

El XGBoost tuneado ya es bueno, pero para exprimir el último 1% de AUC aplicamos:

1. **K-Fold con test averaging** (predicciones promediadas entre folds).
2. **Multi-seed averaging** (3 semillas distintas) — reduce la varianza de cada modelo.
3. **4 modelos diversos**: XGBoost, LightGBM, CatBoost, HistGradientBoosting.
4. **Blend ponderado** con pesos que **maximizan el AUC sobre las predicciones OOF** (out-of-fold) — sin leakage.

Esta es la combinación que produjo el mejor score en la competencia.

In [ ]:
SEEDS = [42, 7, 2024]
N_FOLDS = 5

def get_models(seed):
    return {
        'xgb': XGBClassifier(n_estimators=3000, learning_rate=0.025, max_depth=5,
                              min_child_weight=4, subsample=0.85, colsample_bytree=0.8,
                              reg_alpha=0.3, reg_lambda=1.5, gamma=0.1,
                              eval_metric='auc', random_state=seed, n_jobs=-1,
                              early_stopping_rounds=100, tree_method='hist'),
        'lgb': LGBMClassifier(n_estimators=3000, learning_rate=0.025, num_leaves=64,
                               min_child_samples=20, subsample=0.85, colsample_bytree=0.8,
                               reg_alpha=0.3, reg_lambda=1.5,
                               random_state=seed, n_jobs=-1, verbose=-1),
        'cat': CatBoostClassifier(iterations=3000, learning_rate=0.035, depth=6,
                                   l2_leaf_reg=3.0, eval_metric='AUC',
                                   random_state=seed, verbose=False,
                                   early_stopping_rounds=100),
        'hgb': HistGradientBoostingClassifier(max_iter=1500, learning_rate=0.04,
                                               max_depth=7, min_samples_leaf=20,
                                               l2_regularization=0.5, random_state=seed,
                                               early_stopping=True, validation_fraction=0.15,
                                               n_iter_no_change=50),
    }

model_names = ['xgb','lgb','cat','hgb']
oof  = {m: np.zeros(len(X)) for m in model_names}
test_pred = {m: np.zeros(len(X_test)) for m in model_names}

X_np  = X.values.astype(np.float32)
Xt_np = X_test.values.astype(np.float32)

for seed in SEEDS:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    print(f'\n##### SEED {seed} #####')
    for fold, (tr, va) in enumerate(skf.split(X_np, y)):
        Xtr, Xva, ytr, yva = X_np[tr], X_np[va], y[tr], y[va]
        models = get_models(seed)
        aucs = {}

        m = models['xgb']
        m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
        p = m.predict_proba(Xva)[:,1]
        oof['xgb'][va]   += p / len(SEEDS)
        test_pred['xgb'] += m.predict_proba(Xt_np)[:,1] / (len(SEEDS)*N_FOLDS)
        aucs['xgb'] = roc_auc_score(yva, p)

        m = models['lgb']
        m.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric='auc',
              callbacks=[lgb_early_stop(100, verbose=False)])
        p = m.predict_proba(Xva)[:,1]
        oof['lgb'][va]   += p / len(SEEDS)
        test_pred['lgb'] += m.predict_proba(Xt_np)[:,1] / (len(SEEDS)*N_FOLDS)
        aucs['lgb'] = roc_auc_score(yva, p)

        m = models['cat']
        m.fit(Xtr, ytr, eval_set=(Xva, yva), use_best_model=True, verbose=False)
        p = m.predict_proba(Xva)[:,1]
        oof['cat'][va]   += p / len(SEEDS)
        test_pred['cat'] += m.predict_proba(Xt_np)[:,1] / (len(SEEDS)*N_FOLDS)
        aucs['cat'] = roc_auc_score(yva, p)

        m = models['hgb']
        m.fit(Xtr, ytr)
        p = m.predict_proba(Xva)[:,1]
        oof['hgb'][va]   += p / len(SEEDS)
        test_pred['hgb'] += m.predict_proba(Xt_np)[:,1] / (len(SEEDS)*N_FOLDS)
        aucs['hgb'] = roc_auc_score(yva, p)

        print(f'  Fold {fold+1}: ' + '  '.join(f'{k}={v:.5f}' for k,v in aucs.items()))

print('\n=== OOF AUC por modelo (promediado entre seeds) ===')
for k in model_names:
    print(f'  {k:4s}: {roc_auc_score(y, oof[k]):.5f}')

In [ ]:
# Búsqueda de pesos óptimos del blend sobre las predicciones OOF
best_auc, best_w = 0, None
step = 0.05
for wx in np.arange(0, 1.001, step):
    for wl in np.arange(0, 1.001-wx, step):
        for wc in np.arange(0, 1.001-wx-wl, step):
            wh = 1 - wx - wl - wc
            if wh < -1e-9: continue
            wh = max(wh, 0)
            blend = wx*oof['xgb'] + wl*oof['lgb'] + wc*oof['cat'] + wh*oof['hgb']
            a = roc_auc_score(y, blend)
            if a > best_auc:
                best_auc, best_w = a, (wx, wl, wc, wh)
wx, wl, wc, wh = best_w
print(f'Pesos blend óptimos -> XGB={wx:.2f}  LGB={wl:.2f}  CAT={wc:.2f}  HGB={wh:.2f}')
print(f'AUC blend (OOF): {best_auc:.5f}')

blend_oof = wx*oof['xgb'] + wl*oof['lgb'] + wc*oof['cat'] + wh*oof['hgb']
fpr, tpr, _ = roc_curve(y, blend_oof)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f'Blend AUC = {best_auc:.4f}')
plt.plot([0,1],[0,1],'--', color='gray')
plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('Curva ROC del ensemble final (OOF)')
plt.legend(); plt.show()

## 7. Generación de `submission.csv`

Aplicamos los pesos óptimos sobre las predicciones promediadas del test (que ya integran 5 folds × 3 seeds × 4 modelos = 60 predicciones por fila) y guardamos el archivo en el formato que pide Kaggle: `id, Exited` (probabilidad).

In [ ]:
final_test = wx*test_pred['xgb'] + wl*test_pred['lgb'] + wc*test_pred['cat'] + wh*test_pred['hgb']

submission = pd.DataFrame({'id': test_ids, 'Exited': final_test})
submission.to_csv('submission.csv', index=False)
print('submission.csv shape:', submission.shape)
submission.head()

## 8. Conclusiones

### Qué funcionó

- **Feature engineering** (ratios, interacciones, bins, frequency / target encoding) aportó la mayor mejora individual. Pasar de ~50 features brutas a ~60 features ingeniadas movió el AUC notablemente.
- Los **gradient boosting** (XGBoost, LightGBM, CatBoost) y HistGradientBoosting son superiores a Logistic Regression y Random Forest en este problema tabular.
- **K-Fold con test averaging** reduce la varianza de la predicción y suele sumar +0.002 ~ +0.005 AUC frente a un único split.
- **Multi-seed averaging** (3 semillas) suaviza las predicciones de los modelos estocásticos.
- **Blend ponderado optimizando sobre OOF** es más robusto que la media simple, sin leakage. En nuestro caso, **CatBoost** se llevó el mayor peso.

### Progresión de scores

| Iteración | Estrategia | OOF AUC |
|-----------|-----------|---------|
| v1 | XGBoost tuneado con RandomizedSearchCV | ~0.934 |
| v2 | Blend XGB+LGB+CAT + feature engineering básico | ~0.935 |
| **v3 (final)** | Multi-seed + 4 modelos + FE extendido + target encoding | **~0.937** |

### Posibles mejoras adicionales

- **Stacking** con meta-modelo (Logistic Regression / Ridge) sobre las OOF predictions.
- **Optuna** para tuning fino de cada modelo individual.
- **Pseudo-labeling** con las predicciones más confiables del test set.
- Concatenar el **dataset original** (`Churn_Modelling.csv`) como datos de entrenamiento extra (técnica habitual de los top en este tipo de competencias).